# MoXpert — Router Network (paper Eq. 4-5)

`p = sigmoid(MLP(V_fuse))`

The **Router Network** is the gate that decides, for a given (query image, query
text), **which experts to activate**. Unlike sparse-LLM MoE (homogeneous FFN
experts + a softmax over a simplex), MoXpert has **4 heterogeneous functional
experts** and the router emits an **independent activation probability per
expert** — a *multi-label* sigmoid, **not** a softmax.

| Paper | Equation | Where in this notebook |
|-------|----------|------------------------|
| Eq. 1-3 | `V_img`, `V_text`, `V_fuse = [V_img ; V_text]` | *optional* CLIP encoder cell |
| **Eq. 4** | **`p = sigmoid(MLP(V_fuse))`,  `p ∈ [0,1]^N`** | `RouterMLP` |
| **Eq. 5** | BCE training of the router | training cell |

- `V_fuse` is **1152-d** = `2 × 576` (per-modality `d = 576`).
- `N = 4` experts: **Reference Extractor, Knowledge Guide, Reasoning Expert, Decision Maker**.
- The binary activation decision uses a **per-expert threshold** `τ_i`
  (default `0.5`): `y_i = 1 if p_i ≥ τ_i else 0`.

**Runs on both** local macOS (MPS/CPU) **and** Google Colab / cloud GPU (CUDA).
This notebook is **self-contained** — it defines everything inline and needs only
`torch`, `numpy`, `scikit-learn` for the core + verification. The real CLIP encoder
(Eq. 1-3) is an **optional** cell that self-skips if `clip` is not installed.


## 1. Environment detection & dependencies

Detects Colab vs local and installs only what's missing. Core cells are dependency-light so the verification suite always runs.

In [ ]:
# --- Environment detection (Colab vs local macOS / cloud GPU) ---
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Colab: {IN_COLAB}")

# Core deps for Eq. 4-5 + verification: torch, numpy, scikit-learn.
# On Colab these are usually preinstalled; install defensively there only.
if IN_COLAB:
    import importlib, subprocess, sys
    for pkg, mod in [("numpy", "numpy"), ("scikit-learn", "sklearn"), ("torch", "torch")]:
        if importlib.util.find_spec(mod) is None:
            print(f"Installing {pkg} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
# On local macOS we assume the project venv already has torch/numpy/scikit-learn.
# If not, run:  pip install torch numpy scikit-learn

## 2. Device auto-select (`cuda` → `mps` → `cpu`)

Prioritises a cloud GPU (CUDA) when present, falls back to Apple MPS on a Mac, then CPU. Same ordering as `Experiments/Qwen2-VL.py`.

In [ ]:
import numpy as np
import torch

def select_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = select_device()
print(f"torch  : {torch.__version__}")
print(f"device : {DEVICE}")

## 3. Expert definitions (paper Sec. 3)

The four heterogeneous experts, in canonical order (the router's output dimension
follows this order):

1. **Reference Extractor** — retrieves the most similar *normal* image (visual grounding).
2. **Knowledge Guide** — injects object-specific domain knowledge.
3. **Reasoning Expert** — adds an explicit Chain-of-Thought scaffold.
4. **Decision Maker** — always-on synthesiser that forces the final answer.


In [ ]:
# --- Expert constants (canonical order == router output order) ---
REFERENCE_EXTRACTOR = "Reference Extractor"
KNOWLEDGE_GUIDE     = "Knowledge Guide"
REASONING_EXPERT    = "Reasoning Expert"
DECISION_MAKER      = "Decision Maker"
EXPERT_NAMES = [REFERENCE_EXTRACTOR, KNOWLEDGE_GUIDE, REASONING_EXPERT, DECISION_MAKER]
N_EXPERTS = len(EXPERT_NAMES)


def activation_vector(active) -> np.ndarray:
    """Binary activation vector y ∈ {0,1}^N for the given active expert names."""
    active = set(active)
    return np.array([1.0 if name in active else 0.0 for name in EXPERT_NAMES],
                    dtype=np.float64)


# --- Per-expert decision threshold τ_i (Eq. 4 -> binary decision) ---
def _as_tau_vector(tau) -> np.ndarray:
    """Broadcast/validate a threshold into a per-expert vector of length N_EXPERTS.

    Accepts a scalar (same τ for all experts) OR a length-N list / np.ndarray /
    torch.Tensor, so each expert's τ_i can be tuned independently later,
    e.g. [0.4, 0.6, 0.5, 0.5].
    """
    if isinstance(tau, torch.Tensor):
        tau = tau.detach().cpu().numpy()
    tau = np.asarray(tau, dtype=np.float64)
    if tau.ndim == 0:                      # scalar -> broadcast
        tau = np.full(N_EXPERTS, float(tau))
    if tau.shape != (N_EXPERTS,):
        raise ValueError(f"tau must be scalar or shape ({N_EXPERTS},), got {tau.shape}")
    return tau


def apply_threshold(probs, tau=0.5) -> np.ndarray:
    """Convert sigmoid probabilities p_i -> binary decisions y_i.

        y_i = 1  if p_i >= tau_i   else   0

    `probs` may be shape (N_EXPERTS,) or batched (n, N_EXPERTS).
    `tau` is a scalar (default 0.5 for every expert) or a per-expert vector.
    Returns an int array of the same shape as `probs`.
    """
    if isinstance(probs, torch.Tensor):
        probs = probs.detach().cpu().numpy()
    probs = np.asarray(probs, dtype=np.float64)
    tau_vec = _as_tau_vector(tau)          # (N_EXPERTS,) -> broadcasts over rows
    return (probs >= tau_vec).astype(int)


def experts_from_vector(vec, tau=0.5):
    """Names of experts whose (per-expert-thresholded) activation is 1."""
    y = apply_threshold(np.asarray(vec, dtype=np.float64).reshape(-1), tau)
    return [name for name, v in zip(EXPERT_NAMES, y) if v == 1]


# Default thresholds: τ_i = 0.5 for every expert (tune independently later).
DEFAULT_TAU = np.full(N_EXPERTS, 0.5)
print("Experts       :", EXPERT_NAMES)
print("Default tau   :", DEFAULT_TAU.tolist())
print("Example custom:", _as_tau_vector([0.4, 0.6, 0.5, 0.5]).tolist())

## 4. Router Network — Eq. 4  `p = sigmoid(MLP(V_fuse))`

A small MLP maps the fused embedding to per-expert **logits**, then a **sigmoid**
gives independent activation probabilities `p ∈ [0,1]^N` (multi-label, not a
softmax simplex). Default architecture: `1152 → 512 → 256 → 4`, ReLU + dropout 0.2.

- `logits(x)` — raw pre-sigmoid outputs, used by `BCEWithLogitsLoss` (Eq. 5).
- `predict_proba(x)` — numpy convenience; runs in **eval mode** (dropout off) so
  it is deterministic. `__call__` on a numpy array routes here (SHAP-friendly).


In [ ]:
from typing import List, Optional
from torch import nn


class RouterMLP(nn.Module):
    """Multi-label gating MLP: V_fuse (1152) -> p (N), p = sigmoid(MLP(V_fuse))."""

    def __init__(self, in_dim: int = 1152, n_experts: int = N_EXPERTS,
                 hidden=(512, 256), dropout: float = 0.2) -> None:
        super().__init__()
        self.in_dim = int(in_dim)
        self.n_experts = int(n_experts)
        self.hidden = tuple(hidden)
        self.dropout = float(dropout)

        layers: List[nn.Module] = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_experts))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Eq. 4: activation probabilities sigmoid(logits)."""
        return torch.sigmoid(self.net(x))

    def logits(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    # -- numpy convenience (deterministic: eval + no_grad) ---------------
    def predict_proba(self, v_fuse: np.ndarray) -> np.ndarray:
        self.eval()
        device = next(self.parameters()).device
        x = torch.as_tensor(np.atleast_2d(v_fuse), dtype=torch.float32, device=device)
        with torch.no_grad():
            return self.forward(x).cpu().numpy()

    def __call__(self, x):  # keep numpy arrays on the deterministic path
        if isinstance(x, np.ndarray):
            return self.predict_proba(x)
        return super().__call__(x)


# Smoke check: build a router and run one forward pass.
_router = RouterMLP(in_dim=1152, n_experts=N_EXPERTS).to(DEVICE)
_p = _router(np.zeros((1, 1152), dtype=np.float32))
print(f"RouterMLP ready: in_dim=1152 -> p shape {_p.shape}, p={np.round(_p, 3).tolist()}")

## 5. Training — Eq. 5 (multi-label BCE)

The router is trained with **`BCEWithLogitsLoss`** (independent per-expert BCE),
using a per-expert **`pos_weight`** (inverse positive frequency) to counter class
imbalance. Optimiser: Adam, `lr=1e-3`, `weight_decay=1e-4`.

`select_threshold` then grid-searches a **single** τ on the eval split to maximise
macro-F1 (a convenience starting point; the per-expert `apply_threshold` above lets
you refine each `τ_i` independently afterwards).

In [ ]:
from typing import Tuple


def train_router(X, Y, is_train, hidden=(512, 256), dropout=0.2, epochs=200,
                 lr=1e-3, weight_decay=1e-4, batch_size=64, seed=0,
                 device=None, verbose=True):
    """Train a RouterMLP with BCE (Eq. 5). Returns (model, loss_history)."""
    torch.manual_seed(seed)
    device = device or DEVICE

    Xtr = torch.tensor(X[is_train], dtype=torch.float32, device=device)
    Ytr = torch.tensor(Y[is_train], dtype=torch.float32, device=device)
    model = RouterMLP(in_dim=X.shape[1], n_experts=Y.shape[1],
                      hidden=hidden, dropout=dropout).to(device)

    pos = Ytr.mean(dim=0).clamp(1e-3, 1 - 1e-3)          # per-expert positive rate
    pos_weight = ((1 - pos) / pos).to(device)             # inverse-frequency weighting
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    n = Xtr.shape[0]
    history: List[float] = []
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        epoch_loss = 0.0
        for s in range(0, n, batch_size):
            idx = perm[s:s + batch_size]
            opt.zero_grad()
            loss = criterion(model.logits(Xtr[idx]), Ytr[idx])
            loss.backward()
            opt.step()
            epoch_loss += float(loss.detach()) * idx.numel()
        history.append(epoch_loss / max(1, n))
        if verbose and (epoch % 25 == 0 or epoch == epochs - 1):
            print(f"[train] epoch {epoch:3d}  BCE={history[-1]:.4f}")
    return model, history


def macro_f1(y_true, y_pred) -> float:
    """Macro-averaged F1 over experts, computed from BINARY decisions."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    f1s = []
    for j in range(y_true.shape[1]):
        tp = np.sum((y_pred[:, j] == 1) & (y_true[:, j] == 1))
        fp = np.sum((y_pred[:, j] == 1) & (y_true[:, j] == 0))
        fn = np.sum((y_pred[:, j] == 0) & (y_true[:, j] == 1))
        denom = 2 * tp + fp + fn
        f1s.append(1.0 if denom == 0 else (2 * tp) / denom)
    return float(np.mean(f1s))


def select_threshold(model, X, Y, is_train) -> Tuple[float, float]:
    """Grid-search a single τ on the eval split, maximising macro-F1."""
    eval_mask = ~is_train
    if eval_mask.sum() == 0:
        eval_mask = np.ones(len(is_train), dtype=bool)
    probs = model.predict_proba(X[eval_mask])
    ytrue = Y[eval_mask]
    best_tau, best_f1 = 0.5, -1.0
    for tau in np.linspace(0.1, 0.9, 33):
        f1 = macro_f1(ytrue, apply_threshold(probs, float(tau)))
        if f1 > best_f1:
            best_f1, best_tau = f1, float(tau)
    return best_tau, best_f1


# --- Deterministic synthetic dataset with a KNOWN linear rule ----------
# The router must learn to recover per-expert activations from V_fuse, so a
# fixed random linear teacher generates the labels; Decision Maker is always-on.
def make_synthetic(n=1200, d=1152, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, d)).astype(np.float32)
    W = rng.standard_normal((d, N_EXPERTS)).astype(np.float32) / np.sqrt(d)
    b = rng.standard_normal(N_EXPERTS).astype(np.float32) * 0.1
    logits = X @ W + b
    Y = (logits > 0).astype(np.float32)
    Y[:, EXPERT_NAMES.index(DECISION_MAKER)] = 1.0        # Decision Maker always active
    is_train = np.zeros(n, dtype=bool)
    is_train[: int(0.7 * n)] = True                       # 70% train / 30% eval
    return X, Y, is_train


print("Training utilities ready (train_router, macro_f1, select_threshold, make_synthetic).")

## 6. *(Optional)* Real CLIP encoder — Eq. 1-3

Builds a **real** 1152-d `V_fuse` from an image + question using a **frozen**
CLIP ViT-B/16, with deterministic frozen `512 → 576` projections per modality
(paper `d = 576`), then `V_fuse = [V_img ; V_text]`.

> **Requires** the `clip` package (`pip install git+https://github.com/openai/CLIP.git`).
> This cell **self-skips** with a message if `clip` is unavailable — the core
> Router (Eq. 4-5) and the verification suite do **not** need it.

In [ ]:
# Optional end-to-end path: real V_fuse from CLIP. Self-skips if clip is absent.
CLIP_NATIVE_DIM = 512   # OpenAI CLIP ViT-B/16 embedding size
PAPER_DIM       = 576   # paper's per-modality d

class MoXpertEncoder:
    """Frozen CLIP ViT-B/16 + fixed 512->576 projections -> V_fuse (1152)."""

    def __init__(self, clip_model_name="ViT-B/16", target_dim=PAPER_DIM,
                 device=None, proj_seed=1234):
        import clip
        self._clip = clip
        self.device = device or DEVICE
        self.model, self.preprocess = clip.load(clip_model_name, device=self.device)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.native_dim = int(self.model.visual.output_dim)
        self.target_dim = int(target_dim)
        g = torch.Generator().manual_seed(proj_seed)
        self.img_proj = self._make_projection(self.native_dim, self.target_dim, g)
        self.text_proj = self._make_projection(self.native_dim, self.target_dim, g)

    def _make_projection(self, in_dim, out_dim, gen):
        if in_dim == out_dim:
            return nn.Identity()
        proj = nn.Linear(in_dim, out_dim, bias=False)
        with torch.no_grad():
            w = torch.empty(out_dim, in_dim)
            w.normal_(0.0, 1.0 / np.sqrt(in_dim), generator=gen)
            proj.weight.copy_(w)
        for p in proj.parameters():
            p.requires_grad_(False)
        return proj.to(self.device)

    def _to_pil(self, image):
        from PIL import Image
        if hasattr(image, "size") and not hasattr(image, "ndim"):
            return image
        if isinstance(image, (str, bytes)):
            return Image.open(image).convert("RGB")
        return image

    def encode_image(self, image) -> np.ndarray:
        x = self.preprocess(self._to_pil(image)).unsqueeze(0).to(self.device)
        with torch.no_grad():
            feat = self.model.encode_image(x)
            feat = feat / feat.norm(dim=-1, keepdim=True)
            v = self.img_proj(feat.float())
        return v.detach().cpu().numpy().reshape(-1)

    def encode_text(self, text: str) -> np.ndarray:
        tokens = self._clip.tokenize([text], truncate=True).to(self.device)
        with torch.no_grad():
            feat = self.model.encode_text(tokens)
            feat = feat / feat.norm(dim=-1, keepdim=True)
            v = self.text_proj(feat.float())
        return v.detach().cpu().numpy().reshape(-1)

    @staticmethod
    def fuse(v_img, v_text) -> np.ndarray:
        """Eq. 3: V_fuse = [V_img ; V_text]."""
        return np.concatenate([np.asarray(v_img).reshape(-1),
                               np.asarray(v_text).reshape(-1)])

    def encode(self, image, text: str) -> np.ndarray:
        return self.fuse(self.encode_image(image), self.encode_text(text))


try:
    import clip  # noqa: F401
    _HAS_CLIP = True
except ImportError:
    _HAS_CLIP = False

if _HAS_CLIP:
    from PIL import Image
    encoder = MoXpertEncoder(device=DEVICE)
    demo_img = Image.new("RGB", (224, 224), (127, 127, 127))   # placeholder grey image
    v_fuse = encoder.encode(demo_img, "Is there a defect in this object?")
    print(f"V_fuse shape: {v_fuse.shape} (expect (1152,))")
    p = _router(v_fuse[None, :].astype(np.float32))
    print("Router p on real V_fuse:", np.round(p, 3).tolist())
    print("Active experts (tau=0.5):", experts_from_vector(p[0], tau=0.5))
else:
    print("[skip] `clip` not installed -> optional CLIP encoder cell skipped.")
    print("       Install with: pip install git+https://github.com/openai/CLIP.git")

## 7. Verification — does it behave as the paper specifies?

Self-contained test suite (written fresh in this cell — nothing imported from the
project's test files). Each check prints `PASS`/`FAIL`; a summary prints at the end.

Checks:
1. **Shape** — probability `p` and binary decision `y` are both `(n, 4)`.
2. **Range** — every `p ∈ [0, 1]`.
3. **Multi-label ≠ softmax** — rows of `p` do **not** sum to 1.
4. **Deterministic (eval)** — repeated `predict_proba` calls match (dropout off).
5. **Save/Load `state_dict`** — round-trip preserves outputs.
6. **BCE decreases** — training loss drops meaningfully.
7. **F1 > 0.6** — macro-F1 from **binary decisions `y_i`** on synthetic eval data.

In [ ]:
# =========================================================================
# Self-contained verification suite (authored fresh; no external test imports)
# =========================================================================
results = []
def check(name, passed):
    results.append((name, bool(passed)))
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

torch.manual_seed(0)
np.random.seed(0)

# ---- 1 & 2: shape + range, for probability p AND binary decision y ------
net = RouterMLP(in_dim=1152, n_experts=N_EXPERTS).to(DEVICE)
Xb = np.random.randn(5, 1152).astype(np.float32)
p = net.predict_proba(Xb)
y = apply_threshold(p, tau=DEFAULT_TAU)                     # per-expert τ (default 0.5)
check("shape: p is (5, 4)", p.shape == (5, N_EXPERTS))
check("shape: binary y is (5, 4)", y.shape == (5, N_EXPERTS))
check("dims: in_dim==2*576==1152 and n_experts==4",
      net.in_dim == 2 * 576 == 1152 and net.n_experts == 4)
check("range: all p in [0, 1]", float(p.min()) >= 0.0 and float(p.max()) <= 1.0)
check("binary: y in {0, 1}", set(np.unique(y)).issubset({0, 1}))

# ---- 3: multi-label (sigmoid) != softmax --------------------------------
# Independent sigmoids: row sums are generally != 1. A softmax would force == 1.
row_sums = p.sum(axis=1)
check("multi-label != softmax (row sums != 1)",
      np.any(np.abs(row_sums - 1.0) > 1e-3))

# ---- per-expert threshold behaves independently -------------------------
# Raising one expert's τ to ~1 must zero out that column's decisions.
tau_custom = np.array([0.5, 0.5, 0.5, 0.999])
y_custom = apply_threshold(p, tau=tau_custom)
check("per-expert tau: raising tau_3 -> col 3 all zero",
      bool(np.all(y_custom[:, 3] == 0)))

# ---- 4: deterministic in eval mode (dropout disabled) -------------------
net_do = RouterMLP(in_dim=1152, n_experts=N_EXPERTS, dropout=0.5).to(DEVICE)
p1 = net_do.predict_proba(Xb)
p2 = net_do.predict_proba(Xb)
check("deterministic in eval (dropout=0.5)", np.allclose(p1, p2, atol=1e-6))

# ---- 5: save / load state_dict round-trip -------------------------------
import io
buf = io.BytesIO()
torch.save(net.state_dict(), buf)
buf.seek(0)
net_reloaded = RouterMLP(in_dim=1152, n_experts=N_EXPERTS).to(DEVICE)
net_reloaded.load_state_dict(torch.load(buf, map_location=DEVICE))
check("save/load state_dict preserves outputs",
      np.allclose(net.predict_proba(Xb), net_reloaded.predict_proba(Xb), atol=1e-6))

# ---- 6 & 7: BCE decreases after training, F1 > 0.6 from binary decisions -
Xs, Ys, is_train = make_synthetic(n=1200, d=1152, seed=0)
model, history = train_router(Xs, Ys, is_train, epochs=120, seed=0, verbose=False)
check("BCE loss decreases after training (< 0.8x initial)",
      history[-1] < history[0] * 0.8)

eval_mask = ~is_train
probs_eval = model.predict_proba(Xs[eval_mask])
y_pred = apply_threshold(probs_eval, tau=DEFAULT_TAU)       # BINARY decisions y_i
f1 = macro_f1(Ys[eval_mask], y_pred)
print(f"       (train BCE {history[0]:.4f} -> {history[-1]:.4f}; eval macro-F1 = {f1:.3f})")
check("macro-F1 > 0.6 on synthetic eval (from binary y_i)", f1 > 0.6)

# ---- Summary ------------------------------------------------------------
n_pass = sum(ok for _, ok in results)
n_total = len(results)
print("\n" + "=" * 56)
overall = "PASS" if n_pass == n_total else "FAIL"
print(f"=== RESULT: {overall}  ({n_pass}/{n_total} checks passed) ===")
print("=" * 56)
if overall != "PASS":
    for name, ok in results:
        if not ok:
            print(f"  FAILED -> {name}")

---
## 8. ทดสอบกับข้อมูลจริง — เทรน Router บน CLIP feature จริง (ระดับ 2)

หัวข้อนี้ต่อยอดจาก Section 1-7 (ซึ่งพิสูจน์แค่ว่า *โค้ดถูกต้องตาม paper*) มาเป็นการ
เทรน `RouterMLP` บน **ฟีเจอร์จริง** ที่ได้จากภาพ DS-MVTec จริง เพื่อดูว่า router
เรียนรู้ที่จะเลือก expert จากภาพ+ข้อความได้จริงหรือไม่ แล้วรายงาน **macro-F1 จริง**
(แยกตามชนิดคำถาม)

**⚠️ ข้อควรระวัง (สำคัญ):** ป้ายกำกับ (label) ในระดับนี้สร้างจากกฎ **heuristic**
ที่ผูกกับ `question_type` โดยตรง (ดูตาราง `HEURISTIC_PRIORS` ด้านล่าง) ดังนั้น
ค่า F1 ที่ได้ *น่าจะสูงอยู่แล้ว* — มันพิสูจน์เพียงว่า "ฟีเจอร์ CLIP แยกกลุ่มการ
routing ได้" ไม่ได้พิสูจน์ว่า "router ทำให้โมเดลตอบแม่นขึ้น" คำตอบเรื่องความแม่น
จริงต้องไปวัดที่ **ระดับ 3 (end-to-end กับ Qwen2-VL)** ต้องรันใน env ที่มี `clip`
เช่น `/opt/miniconda3/envs/moxpert/bin/python`


### 8.1 ตั้งค่า path + สร้าง label ด้วยกฎ heuristic (พอร์ตจาก `moxpert/labeling.py`)

In [ ]:
import os, json
from collections import defaultdict
from pathlib import Path

# --- ค้นหา repo root อัตโนมัติ: ไต่ขึ้นไปจนเจอโฟลเดอร์ที่มี Annotation/DS-MVTec.json ---
def find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "Annotation" / "DS-MVTec.json").exists():
            return cand
    raise FileNotFoundError("หา repo root ไม่เจอ (ต้องมี Annotation/DS-MVTec.json)")

REPO = find_repo_root()
ANN_PATH   = REPO / "Annotation" / "DS-MVTec.json"   # ไฟล์คำถาม/คำตอบ
DATA_ROOT  = REPO / "Dataset" / "MMAD"                # โฟลเดอร์รูป (เป็น symlink ไปที่จริง)
ART_DIR    = REPO / "Router_Network" / "artifacts"    # ที่เก็บ feature cache + โมเดล
ART_DIR.mkdir(parents=True, exist_ok=True)
print("REPO      :", REPO)
print("ANN_PATH  :", ANN_PATH, "->", ANN_PATH.exists())
print("DATA_ROOT :", DATA_ROOT, "->", DATA_ROOT.exists())

# --- ตารางกฎ heuristic: question_type -> รายชื่อ expert ที่ควรเปิด ---
# (พอร์ตตรงจาก test1:moxpert/labeling.py ; Decision Maker เปิดเสมอ)
HEURISTIC_PRIORS = {
    "Anomaly Detection":      [REFERENCE_EXTRACTOR, DECISION_MAKER],
    "Anomaly Discrimination": [REFERENCE_EXTRACTOR, DECISION_MAKER],
    "Defect Classification":  [REFERENCE_EXTRACTOR, KNOWLEDGE_GUIDE, DECISION_MAKER],
    "Defect Localization":    [REFERENCE_EXTRACTOR, DECISION_MAKER],
    "Defect Description":     [REFERENCE_EXTRACTOR, KNOWLEDGE_GUIDE, REASONING_EXPERT, DECISION_MAKER],
    "Defect Analysis":        [KNOWLEDGE_GUIDE, REASONING_EXPERT, DECISION_MAKER],
}
_DEFAULT_PRIOR = [REFERENCE_EXTRACTOR, DECISION_MAKER]   # ใช้เมื่อไม่รู้จัก question_type

# จำกัดจำนวนรูปเพื่อทดสอบเร็ว ๆ ก่อน (ตั้ง None = ใช้ครบทั้ง 4,090 รูป)
SAMPLE_LIMIT = 300

def build_heuristic_records(ann_path, data_root, sample_limit=None):
    """อ่าน annotation แล้วสร้าง record ต่อ (รูป x คำถาม) พร้อม label y ตามกฎ heuristic.

    คืนค่า list ของ dict: {image_path, question, question_type, object, y}
    โดย y คือเวกเตอร์ 0/1 ยาว 4 ตามลำดับ EXPERT_NAMES
    """
    with open(ann_path, "r") as f:
        data = json.load(f)
    records = []
    skipped = 0   # นับรูปที่อ้างถึงใน annotation แต่ไม่มีไฟล์บนดิสก์
    for i, (key, item) in enumerate(data.items()):
        if sample_limit is not None and i >= sample_limit:
            break
        object_name = key.split("/")[1] if "/" in key else key   # เช่น "bottle"
        img_path = str(Path(data_root) / key)                    # path รูปจริงแบบเต็ม
        if not os.path.exists(img_path):                         # ข้ามรูปที่หายไป (เช่น good บางรูป)
            skipped += 1
            continue
        for conv in item.get("conversation", []):
            qtype = conv.get("type", "")
            prior = HEURISTIC_PRIORS.get(qtype, _DEFAULT_PRIOR)   # เลือก expert ตามชนิดคำถาม
            records.append({
                "image_path": img_path,
                "question": conv.get("Question", ""),
                "question_type": qtype,
                "object": object_name,
                "y": activation_vector(prior),                   # -> เวกเตอร์ 0/1 ยาว 4
            })
    if skipped:
        print(f"(ข้าม {skipped} รูปที่ไม่มีไฟล์บนดิสก์)")
    return records

records = build_heuristic_records(ANN_PATH, DATA_ROOT, SAMPLE_LIMIT)
print(f"\nสร้าง record ได้ {len(records)} รายการ (จาก {SAMPLE_LIMIT or 'ทั้งหมด'} รูป)")
# นับจำนวนต่อชนิดคำถาม เพื่อดูการกระจายตัว
_cnt = defaultdict(int)
for r in records:
    _cnt[r["question_type"]] += 1
for t, c in sorted(_cnt.items()):
    print(f"  {t:24s}: {c}")

### 8.2 แปลงรูป+ข้อความจริงเป็น `V_fuse` ด้วย CLIP แล้ว cache ไว้

เข้ารหัสรูปที่ไม่ซ้ำแต่ละรูปครั้งเดียว (แคชไว้) เพื่อความเร็ว แล้วประกอบเป็น `X (n,1152)`, `Y (n,4)` พร้อมแบ่ง train/eval แบบ stratified ต่อ (question_type × object) — ผลลัพธ์เก็บลง `features_real.npz`

In [ ]:
NPZ_PATH = ART_DIR / "features_real.npz"

def stratified_is_train(records, train_frac=0.7, seed=0):
    """แบ่ง train/eval แบบ stratified ต่อเซลล์ (question_type x object).

    คืน mask boolean ยาวเท่าจำนวน record : True = อยู่ชุด train
    (พอร์ตแนวคิดจาก test1:moxpert/labeling.py::stratified_split)
    """
    rng = np.random.default_rng(seed)
    cells = defaultdict(list)
    for i, r in enumerate(records):
        cells[(r["question_type"], r["object"])].append(i)   # จัดกลุ่มตามเซลล์
    is_train = np.zeros(len(records), dtype=bool)
    for _, idxs in cells.items():
        idxs = list(idxs)
        rng.shuffle(idxs)
        n_train = max(1, int(round(train_frac * len(idxs)))) if len(idxs) > 1 else 1
        for i in idxs[:n_train]:
            is_train[i] = True
    return is_train

def encode_records(records, encoder):
    """แปลงทุก record เป็นฟีเจอร์: X=V_fuse(1152), Y=label(4), qtypes=ชนิดคำถาม.

    - เข้ารหัสรูปที่ไม่ซ้ำครั้งเดียว (img_cache) และคำถามที่ไม่ซ้ำครั้งเดียว (txt_cache)
    - V_fuse = concat[V_img ; V_text] (Eq. 3)
    """
    img_cache, txt_cache = {}, {}
    X, Y, qtypes = [], [], []
    for i, r in enumerate(records):
        if i % 200 == 0:
            print(f"  encoding {i}/{len(records)} ...")
        ip = r["image_path"]
        if ip not in img_cache:                       # เข้ารหัสรูปครั้งเดียวต่อรูป
            img_cache[ip] = encoder.encode_image(ip)  # -> V_img (576)
        q = r["question"]
        if q not in txt_cache:                        # เข้ารหัสคำถามครั้งเดียวต่อข้อความ
            txt_cache[q] = encoder.encode_text(q)     # -> V_text (576)
        X.append(encoder.fuse(img_cache[ip], txt_cache[q]))   # -> V_fuse (1152)
        Y.append(r["y"])
        qtypes.append(r["question_type"])
    return (np.asarray(X, dtype=np.float32),
            np.asarray(Y, dtype=np.float32),
            np.asarray(qtypes, dtype=object))

# ถ้ามี cache อยู่แล้วให้โหลดเลย ไม่ต้องเข้ารหัสใหม่ (ประหยัดเวลา)
if NPZ_PATH.exists():
    d = np.load(NPZ_PATH, allow_pickle=True)
    X_real, Y_real, qtypes_real, is_train_real = d["X"], d["Y"], d["qtypes"], d["is_train"]
    print(f"โหลด cache: X={X_real.shape}, Y={Y_real.shape}")
elif _HAS_CLIP:
    encoder_real = MoXpertEncoder(device=DEVICE)      # CLIP ViT-B/16 (แช่แข็ง)
    X_real, Y_real, qtypes_real = encode_records(records, encoder_real)
    is_train_real = stratified_is_train(records, train_frac=0.7, seed=0)
    np.savez(NPZ_PATH, X=X_real, Y=Y_real, qtypes=qtypes_real, is_train=is_train_real)
    print(f"เข้ารหัสเสร็จ: X={X_real.shape}, Y={Y_real.shape}  (เซฟที่ {NPZ_PATH})")
    print(f"train={int(is_train_real.sum())}  eval={int((~is_train_real).sum())}")
else:
    # ไม่มี clip -> ข้ามระดับ 2 (ต้องรันใน env moxpert)
    X_real = None
    print("[ข้าม] ไม่พบแพ็กเกจ `clip` -> รันหัวข้อ 8 ไม่ได้")
    print("       ใช้:  /opt/miniconda3/envs/moxpert/bin/python  รันโน้ตบุ๊กนี้")

### 8.3 เทรน Router บนฟีเจอร์จริง + รายงาน macro-F1 แยกตามชนิดคำถาม

In [ ]:
ROUTER_PT = ART_DIR / "router_real.pt"

if X_real is not None:
    # --- เทรน RouterMLP ด้วย BCE (Eq. 5) บนข้อมูลจริง ---
    model_real, hist_real = train_router(
        X_real, Y_real, is_train_real, epochs=150, seed=0, verbose=True)

    # --- เลือก threshold τ ที่ให้ macro-F1 สูงสุดบนชุด eval ---
    tau_real, f1_overall = select_threshold(model_real, X_real, Y_real, is_train_real)
    print(f"\nτ ที่เลือก = {tau_real:.3f} | macro-F1 รวม (eval) = {f1_overall:.3f}")

    # --- คำนวณ macro-F1 แยกตามชนิดคำถาม (จาก binary decision y_i) ---
    eval_mask = ~is_train_real
    probs_eval = model_real.predict_proba(X_real[eval_mask])   # p = sigmoid(...)
    y_pred = apply_threshold(probs_eval, tau=tau_real)         # p -> y (0/1) ที่ τ ที่เลือก
    y_true = Y_real[eval_mask]
    qte    = qtypes_real[eval_mask]

    print("\nmacro-F1 แยกตามชนิดคำถาม (บนชุด eval):")
    print(f"  {'question_type':24s}  {'n':>5s}  {'F1':>6s}")
    for t in sorted(set(qte.tolist())):
        m = (qte == t)
        f1_t = macro_f1(y_true[m], y_pred[m])
        print(f"  {t:24s}  {int(m.sum()):5d}  {f1_t:6.3f}")

    # --- เซฟโมเดล router ที่เทรนกับข้อมูลจริง (ใช้ต่อในระดับ 3) ---
    torch.save({
        "state_dict": model_real.state_dict(),
        "config": {"in_dim": model_real.in_dim, "n_experts": model_real.n_experts,
                   "hidden": model_real.hidden, "dropout": model_real.dropout},
        "expert_names": EXPERT_NAMES,
        "threshold": float(tau_real),
    }, ROUTER_PT)
    print(f"\nเซฟ router -> {ROUTER_PT}")
else:
    print("[ข้าม] ไม่มีฟีเจอร์จริง (ต้องรันเซลล์ 8.2 ใน env ที่มี clip ก่อน)")

### 8.4 อ่านผลอย่างไร

- **macro-F1 รวม** และ **F1 แยกตามชนิดคำถาม** บอกว่า router ทำนายชุด expert ได้ตรง
  กับกฎ heuristic แค่ไหน — ค่ายิ่งใกล้ 1 ยิ่งดี ถ้าต่ำผิดปกติแปลว่ามีบั๊กที่การ
  เข้ารหัสหรือการทำ label
- **τ ที่เลือก** คือ threshold รวมที่ดีที่สุดบน eval — ภายหลังปรับ `τ_i` แยกราย
  expert ได้ด้วย `apply_threshold(p, tau=[...])`
- ไฟล์ที่ได้: `artifacts/features_real.npz` (ฟีเจอร์ที่แคช) และ
  `artifacts/router_real.pt` (โมเดลที่เทรนแล้ว) — จะถูกนำไปใช้ต่อในระดับ 3

> **ย้ำ:** F1 สูงในหัวข้อนี้ *ไม่ได้* แปลว่าโมเดลตอบคำถามแม่นขึ้น เพราะ label ผูกกับ
> `question_type` อยู่แล้ว การพิสูจน์ว่า *improve model* จริงต้องไปที่ระดับ 3
> (เสียบ router เข้า pipeline Qwen2-VL แล้ววัด accuracy จริงเทียบ baseline)

หากต้องการเทรนบนข้อมูล **ครบทั้งหมด** ให้กลับไปตั้ง `SAMPLE_LIMIT = None` ในเซลล์ 8.1
ลบไฟล์ `features_real.npz` เดิม แล้วรันหัวข้อ 8 ใหม่
